# Data Analyst Job Market Analysis

## Day 2 — Data Cleaning

This notebook cleans the raw job-market data collected from the
Arbeitnow Job Board API.

### Objective

Transform the raw API dataset into a structured analytical dataset
suitable for exploratory analysis, PostgreSQL, SQL analysis, and
Power BI.

### Source

Arbeitnow Job Board API

### Raw Records

2,995 job postings

### Important Principle

The original `raw_jobs.json` file is preserved as the raw source layer.
All cleaning and transformations are performed on a separate DataFrame.

In [1]:
import json
import re
import html
import pandas as pd
import numpy as np

In [2]:
RAW_FILE = "../dataset/raw_jobs.json"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

df = pd.DataFrame(raw_data["data"])

print("Raw dataset shape:", df.shape)

Raw dataset shape: (2995, 10)


In [3]:
clean_df = df.copy()

In [4]:
clean_df.isnull().sum()

slug            0
company_name    0
title           0
description     0
remote          0
url             0
tags            0
job_types       0
location        0
created_at      0
dtype: int64

In [5]:
print("Empty tags:", clean_df["tags"].apply(len).eq(0).sum())
print("Empty job types:", clean_df["job_types"].apply(len).eq(0).sum())

Empty tags: 256
Empty job types: 901


In [6]:
for col in ["company_name", "title", "location"]:
    print(col, clean_df[col].str.strip().eq("").sum())

company_name 0
title 0
location 68


In [8]:
clean_df["url"].duplicated().sum()

np.int64(122)

In [9]:
clean_df["slug"].duplicated().sum()

np.int64(122)

In [10]:
clean_df = clean_df.drop_duplicates("slug")

In [11]:
print("Rows after cleaning:", len(clean_df))
print("Duplicate slugs:", clean_df["slug"].duplicated().sum())

Rows after cleaning: 2873
Duplicate slugs: 0


In [12]:
for col in ["company_name", "title", "location"]:
    clean_df[col] = clean_df[col].str.strip()

In [13]:
clean_df["description"] = (
    clean_df["description"]
    .str.replace("<[^>]+>", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [14]:
# Converting Tags & Job Types

clean_df["tags"] = clean_df["tags"].apply(
    lambda x: " | ".join(x)
)


In [15]:
clean_df["job_types"] = clean_df["job_types"].apply(
    lambda x: " | ".join(x)
)

In [16]:
# Converting  Date
clean_df["created_at"] = pd.to_datetime(
    clean_df["created_at"],
    unit="s"
)

In [17]:
clean_df["created_at"].head()

0   2026-08-12 06:40:00
1   2026-10-01 05:55:15
2   2026-10-01 05:55:11
3   2026-10-01 05:55:11
4   2026-10-01 05:55:11
Name: created_at, dtype: datetime64[s]

In [18]:
clean_df["posted_date"] = clean_df["created_at"].dt.date
clean_df["posted_year"] = clean_df["created_at"].dt.year
clean_df["posted_month"] = clean_df["created_at"].dt.month_name()

In [19]:
clean_df.shape

(2873, 13)

In [20]:
clean_df.head()

,slug,company_name,title,description,remote,url,tags,job_types,location,created_at,posted_date,posted_year,posted_month
0,software-engineer-price-comparison-portal-berl...,Preiswecker,Software Engineer,Preiswecker is a price comparison portal in Ge...,False,https://www.preiswecker.com/,Software Engineering | E-Commerce,Full-time,Berlin,2026-08-12 06:40:00,2026-08-12,2026,August
1,senior-product-manager-356253,Verve,Senior Product Manager,&lt;div class=&quot;content-intro&quot;&gt;&lt...,False,https://www.arbeitnow.com/jobs/companies/verve...,Product Management,,,2026-10-01 05:55:15,2026-10-01,2026,October
2,working-student-launch-operations-parsdorf-bav...,isaraerospace,Working Student Launch Operations (m/f/d),&lt;p&gt;&lt;strong&gt;&lt;span lang=&quot;EN-...,False,https://www.arbeitnow.com/jobs/companies/isara...,Mission & Launch Operations,,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October
3,senior-mechanical-engineer-parsdorf-bavaria-38...,isaraerospace,Senior Mechanical Engineer (f/m/d),&lt;p class=&quot;p1&quot;&gt;&lt;strong&gt;Mi...,False,https://www.arbeitnow.com/jobs/companies/isara...,Ground Systems,,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October
4,mes-process-and-system-engineer-parsdorf-bavar...,isaraerospace,MES Process and System Engineer (m/f/d),&lt;p&gt;&lt;strong&gt;Mission Brief&amp;nbsp;...,False,https://www.arbeitnow.com/jobs/companies/isara...,Production | Operations & Quality,,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October


In [21]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 2873 entries, 0 to 2994
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype        
---  ------        --------------  -----        
 0   slug          2873 non-null   str          
 1   company_name  2873 non-null   str          
 2   title         2873 non-null   str          
 3   description   2873 non-null   str          
 4   remote        2873 non-null   bool         
 5   url           2873 non-null   str          
 6   tags          2873 non-null   str          
 7   job_types     2873 non-null   str          
 8   location      2873 non-null   str          
 9   created_at    2873 non-null   datetime64[s]
 10  posted_date   2873 non-null   object       
 11  posted_year   2873 non-null   int32        
 12  posted_month  2873 non-null   str          
dtypes: bool(1), datetime64[s](1), int32(1), object(1), str(9)
memory usage: 283.4+ KB


In [22]:
clean_df.isnull().sum()

slug            0
company_name    0
title           0
description     0
remote          0
url             0
tags            0
job_types       0
location        0
created_at      0
posted_date     0
posted_year     0
posted_month    0
dtype: int64

In [23]:
clean_df.to_csv(
    "../dataset/job_market_clean.csv",
    index=False
)

In [24]:
clean_df["location"] = clean_df["location"].replace("", "Unknown")

In [25]:
clean_df["location"].eq("").sum()

np.int64(0)

In [26]:
clean_df["location"].value_counts().head(10)

location
Berlin            469
Munich            224
Hamburg           157
München           111
Unknown            67
Berlin, Berlin     49
Remote             47
Cologne            44
Stuttgart          44
London             44
Name: count, dtype: int64

In [27]:
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("Duplicate slugs:", clean_df["slug"].duplicated().sum())
print("Missing values:", clean_df.isnull().sum().sum())
print("Blank titles:", clean_df["title"].eq("").sum())
print("Blank companies:", clean_df["company_name"].eq("").sum())
print("Blank locations:", clean_df["location"].eq("").sum())

Rows: 2873
Columns: 13
Duplicate slugs: 0
Missing values: 0
Blank titles: 0
Blank companies: 0
Blank locations: 0


In [28]:
clean_df.to_csv(
    "../dataset/job_market_clean.csv",
    index=False
)

In [29]:
clean_df.shape

(2873, 13)

In [30]:
clean_df.head()

,slug,company_name,title,description,remote,url,tags,job_types,location,created_at,posted_date,posted_year,posted_month
0,software-engineer-price-comparison-portal-berl...,Preiswecker,Software Engineer,Preiswecker is a price comparison portal in Ge...,False,https://www.preiswecker.com/,Software Engineering | E-Commerce,Full-time,Berlin,2026-08-12 06:40:00,2026-08-12,2026,August
1,senior-product-manager-356253,Verve,Senior Product Manager,&lt;div class=&quot;content-intro&quot;&gt;&lt...,False,https://www.arbeitnow.com/jobs/companies/verve...,Product Management,,Unknown,2026-10-01 05:55:15,2026-10-01,2026,October
2,working-student-launch-operations-parsdorf-bav...,isaraerospace,Working Student Launch Operations (m/f/d),&lt;p&gt;&lt;strong&gt;&lt;span lang=&quot;EN-...,False,https://www.arbeitnow.com/jobs/companies/isara...,Mission & Launch Operations,,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October
3,senior-mechanical-engineer-parsdorf-bavaria-38...,isaraerospace,Senior Mechanical Engineer (f/m/d),&lt;p class=&quot;p1&quot;&gt;&lt;strong&gt;Mi...,False,https://www.arbeitnow.com/jobs/companies/isara...,Ground Systems,,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October
4,mes-process-and-system-engineer-parsdorf-bavar...,isaraerospace,MES Process and System Engineer (m/f/d),&lt;p&gt;&lt;strong&gt;Mission Brief&amp;nbsp;...,False,https://www.arbeitnow.com/jobs/companies/isara...,Production | Operations & Quality,,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October


In [31]:
import html

clean_df["description"] = (
    clean_df["description"]
    .apply(html.unescape)
    .str.replace(r"<[^>]+>", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [32]:
clean_df["description"].head(3)

0    Preiswecker is a price comparison portal in Ge...
1    Who We Are Verve has created a more efficient ...
2    Mission Brief As Working Student Launch Operat...
Name: description, dtype: str

In [33]:
print(clean_df["description"].iloc[0])

Preiswecker is a price comparison portal in Germany, helping thousands of users find the best deals online every day. We are looking for a Software Engineer to join our team and help build our price comparison platform. We are currently offering the following position: Software Engineer - Price comparison portal Responsibilities Design, build, and maintain features for our price comparison platform Develop and optimize data pipelines that process merchant price feeds Work closely with Product and BI to ship reliable, scalable features Write clean, well-tested code and participate in code reviews Monitor and improve performance of our systems Qualifications completed degree in computer science or a related field, or equivalent experience solid experience with backend development and relational databases familiarity with modern software engineering practices (testing, CI/CD, version control) analytical mindset and a strong sense of ownership good German and English Benefits flexible work

In [34]:
clean_df.to_csv(
    "../dataset/job_market_clean.csv",
    index=False
)